In [29]:
import pandas as pd
import glob
import os
import re

File Loading

In [3]:
DATA_DIR = "/content/drive/MyDrive/NewestCNCollabNote"

In [23]:
# load & process to filter only collaborative notes

note_files = sorted(glob.glob(os.path.join(DATA_DIR, "notes*.tsv")))
print("Found note files:", note_files)

notes_list = []
for f in note_files:
    df = pd.read_csv(
        f,
        sep="\t",
        usecols=["noteId", "tweetId", "isCollaborativeNote", "createdAtMillis", "summary", "classification"],
        low_memory=False
    )
    notes_list.append(df)

notes_df = pd.concat(notes_list, ignore_index=True)

collab_notes = notes_df[notes_df["isCollaborativeNote"] == 1][["noteId", "tweetId", "createdAtMillis", "summary", "classification"]].drop_duplicates()

print("Total notes rows loaded:", len(notes_df))
print("Unique collaborative noteIds:", collab_notes["noteId"].nunique())

Found note files: ['/content/drive/MyDrive/NewestCNCollabNote/notes-00000.tsv', '/content/drive/MyDrive/NewestCNCollabNote/notes-00001.tsv']
Total notes rows loaded: 2506283
Unique collaborative noteIds: 11906


In [24]:
collab_notes.head()

,noteId,tweetId,createdAtMillis,summary,classification
2494377,2033333592244596856,1261399556702351360,1773619630910,This post solicits testimonials from past clie...,NOT_MISLEADING
2494378,2021411467829051490,1300931249947672576,1770777181530,"Claim that Avicii, Bennington, Cornell, Bourda...",MISINFORMED_OR_POTENTIALLY_MISLEADING
2494379,2026243082576965759,1496720501472780291,1771929103719,2022 post said Ukraine's military disintegrate...,MISINFORMED_OR_POTENTIALLY_MISLEADING
2494380,2023532126180876533,1644068292389228550,1771282801575,The video in this post is fabricated; no such ...,MISINFORMED_OR_POTENTIALLY_MISLEADING
2494381,2024855965028553026,1670899448086446080,1771598441443,"The attack shown occurred in Bordeaux, France ...",MISINFORMED_OR_POTENTIALLY_MISLEADING


In [6]:
# save the processed notes
out_path = os.path.join(DATA_DIR, "collaborative_notes.parquet")
collab_notes.to_parquet(out_path, index=False)
print("Saved:", out_path)

Saved: /content/drive/MyDrive/NewestCNCollabNote/collaborative_notes.parquet


In [7]:
# get ratings with rater id, created time, and the suggest content
rating_files = sorted(glob.glob(os.path.join(DATA_DIR, "ratings*.tsv")))
print("Found note files:", rating_files)

ratings = []
usecols = ["noteId", "raterParticipantId", "createdAtMillis", "suggestion"]  # keep minimal columns for speed

for f in rating_files:
    df = pd.read_csv(f, sep="\t", usecols=lambda c: c in usecols, low_memory=False)
    ratings.append(df)

ratings_df = pd.concat(ratings, ignore_index=True)

# Keep suggestion rows only (non-null and non-empty after strip)
ratings_df["suggestion"] = ratings_df["suggestion"].astype("string")
suggestions_df = ratings_df[ratings_df["suggestion"].notna() & (ratings_df["suggestion"].str.strip() != "")].copy()
suggestions_df["ratingCreatedAt"] = pd.to_datetime(suggestions_df["createdAtMillis"], unit="ms", utc=True)

print("Total ratings rows:", len(ratings_df))
print("Suggestion rows:", len(suggestions_df))
print("Unique noteIds with suggestions:", suggestions_df["noteId"].nunique())

Found note files: ['/content/drive/MyDrive/NewestCNCollabNote/ratings-00000.tsv', '/content/drive/MyDrive/NewestCNCollabNote/ratings-00001.tsv', '/content/drive/MyDrive/NewestCNCollabNote/ratings-00002.tsv', '/content/drive/MyDrive/NewestCNCollabNote/ratings-00003.tsv', '/content/drive/MyDrive/NewestCNCollabNote/ratings-00004.tsv', '/content/drive/MyDrive/NewestCNCollabNote/ratings-00005.tsv', '/content/drive/MyDrive/NewestCNCollabNote/ratings-00006.tsv', '/content/drive/MyDrive/NewestCNCollabNote/ratings-00007.tsv']
Total ratings rows: 205838618
Suggestion rows: 5911
Unique noteIds with suggestions: 4747


In [8]:
suggestions_df.head()

,noteId,raterParticipantId,createdAtMillis,suggestion,ratingCreatedAt
194464437,2014117943958012199,6D24782082A2ACBD43F46DCFF5E4A233CF891065EA1AB3...,1769128077708,please elaborate more on &quot;appears to be a...,2026-01-23 00:27:57.708000+00:00
194465732,2014121093532098991,6D24782082A2ACBD43F46DCFF5E4A233CF891065EA1AB3...,1769117393958,"good note, good job 👍",2026-01-22 21:29:53.958000+00:00
194696750,2014450474209182195,6D24782082A2ACBD43F46DCFF5E4A233CF891065EA1AB3...,1769121328744,Can you add more reliable sources? Thanks.,2026-01-22 22:35:28.744000+00:00
194708795,2014466977717485974,6D24782082A2ACBD43F46DCFF5E4A233CF891065EA1AB3...,1769128425651,please explain more details around &quot;metap...,2026-01-23 00:33:45.651000+00:00
194740992,2014495172550406429,6D24782082A2ACBD43F46DCFF5E4A233CF891065EA1AB3...,1769180881772,"Note is needed, post is obviously misleading",2026-01-23 15:08:01.772000+00:00


In [9]:
# save the processed ratings
out_path = os.path.join(DATA_DIR, "collab_ratings.parquet")
suggestions_df.to_parquet(out_path, index=False)
print("Saved:", out_path)

Saved: /content/drive/MyDrive/NewestCNCollabNote/collab_ratings.parquet


In [10]:
# Inner join ensures we only analyze suggestions attached to collaborative notes
collab_suggestions = suggestions_df.merge(collab_notes, on="noteId", how="inner")
print("Suggestion rows on collaborative notes:", len(collab_suggestions))
print("Unique collaborative notes with suggestions:", collab_suggestions["noteId"].nunique())

Suggestion rows on collaborative notes: 5910
Unique collaborative notes with suggestions: 4746


In [11]:
collab_suggestions.head(2)

,noteId,raterParticipantId,createdAtMillis,suggestion,ratingCreatedAt,tweetId
0,2014117943958012199,6D24782082A2ACBD43F46DCFF5E4A233CF891065EA1AB3...,1769128077708,please elaborate more on &quot;appears to be a...,2026-01-23 00:27:57.708000+00:00,-1
1,2014121093532098991,6D24782082A2ACBD43F46DCFF5E4A233CF891065EA1AB3...,1769117393958,"good note, good job 👍",2026-01-22 21:29:53.958000+00:00,-1


In [12]:
out_path = os.path.join(DATA_DIR, "collab_suggestions.parquet")
collab_suggestions.to_parquet(out_path, index=False)
print("Saved:", out_path)

Saved: /content/drive/MyDrive/NewestCNCollabNote/collab_suggestions.parquet


### Paired notes with tweets

In [25]:
collab_notes["createdAt"] = pd.to_datetime(
    collab_notes["createdAtMillis"], unit="ms", utc=True
)

collab_notes = collab_notes.sort_values(["tweetId", "createdAt", "noteId"]).copy()

# keep only tweets that have multiple collaborative note versions
multi_version = collab_notes.groupby("tweetId")["noteId"].nunique()
multi_version_tweets = multi_version[multi_version > 1].index

version_notes = collab_notes[collab_notes["tweetId"].isin(multi_version_tweets)].copy()

print("tweets with multiple collab versions:", len(multi_version_tweets))
print("unique versioned notes:", version_notes["noteId"].nunique())

tweets with multiple collab versions: 2233
unique versioned notes: 8804


In [26]:
pairs = []

for tweet_id, g in version_notes.groupby("tweetId"):
    g = g.sort_values(["createdAt", "noteId"]).reset_index(drop=True)
    for i in range(len(g) - 1):
        old_row = g.iloc[i]
        new_row = g.iloc[i + 1]
        pairs.append({
            "tweetId": tweet_id,
            "old_noteId": old_row["noteId"],
            "new_noteId": new_row["noteId"],
            "old_createdAt": old_row["createdAt"],
            "new_createdAt": new_row["createdAt"],
        })

version_pairs_df = pd.DataFrame(pairs)
print(version_pairs_df.shape)
version_pairs_df.head()

(6571, 5)


,tweetId,old_noteId,new_noteId,old_createdAt,new_createdAt
0,-1,2014117943958012199,2014121093532098991,2026-01-21 23:29:36.738000+00:00,2026-01-21 23:42:24.674000+00:00
1,-1,2014121093532098991,2014466977717485974,2026-01-21 23:42:24.674000+00:00,2026-01-22 22:42:28.731000+00:00
2,-1,2014466977717485974,2014495284156387774,2026-01-22 22:42:28.731000+00:00,2026-01-23 00:30:03.173000+00:00
3,-1,2014495284156387774,2014495336161829107,2026-01-23 00:30:03.173000+00:00,2026-01-23 00:30:06.576000+00:00
4,-1,2014495336161829107,2014495172550406429,2026-01-23 00:30:06.576000+00:00,2026-01-23 00:34:41.715000+00:00


In [27]:
# only keep the rows with actual suggestion texts
ratings_df["suggestion"] = ratings_df["suggestion"].astype("string")

suggestions_df = ratings_df[
    ratings_df["suggestion"].notna() &
    (ratings_df["suggestion"].str.strip() != "")
].copy()

suggestions_df["ratingCreatedAt"] = pd.to_datetime(
    suggestions_df["createdAtMillis"], unit="ms", utc=True
)

print(suggestions_df.shape)

(5911, 5)


### Identify Version Changes

In [22]:
# attach the old version's suggestion text to the pair
pair_suggestions = version_pairs_df.merge(
    suggestions_df,
    left_on="old_noteId",
    right_on="noteId",
    how="inner"
)

print("total (version, suggestion) pairs:", len(pair_suggestions))
pair_suggestions[["tweetId", "old_noteId", "new_noteId", "suggestion"]].head()

total (version, suggestion) pairs: 4715


,tweetId,old_noteId,new_noteId,suggestion
0,-1,2014117943958012199,2014121093532098991,please elaborate more on &quot;appears to be a...
1,-1,2014121093532098991,2014466977717485974,"good note, good job 👍"
2,-1,2014466977717485974,2014495284156387774,please explain more details around &quot;metap...
3,-1,2014495172550406429,2014496979053879586,"Note is needed, post is obviously misleading"
4,-1,2014496979053879586,2014716760490901695,"This should be a misleading post, context needed"


In [28]:
NOTE_TEXT_COL = "summary"

note_fields = collab_notes[[
    "noteId", "tweetId", NOTE_TEXT_COL,
    "classification"
]].copy()
note_fields = note_fields.rename(columns={
    NOTE_TEXT_COL: "note_text"
})

old_notes = note_fields.rename(columns={
    "noteId": "old_noteId",
    "note_text": "old_note_text",
    "classification": "old_classification"
})

new_notes = note_fields.rename(columns={
    "noteId": "new_noteId",
    "note_text": "new_note_text",
    "classification": "new_classification"
})

pair_suggestions = pair_suggestions.merge(
    old_notes[["old_noteId", "old_note_text", "old_classification"]],
    on="old_noteId",
    how="left"
).merge(
    new_notes[["new_noteId", "new_note_text", "new_classification"]],
    on="new_noteId",
    how="left"
)

In [36]:
pair_suggestions["text_changed"] = (
    pair_suggestions["old_note_text"].fillna("") !=
    pair_suggestions["new_note_text"].fillna("")
)

pair_suggestions["classification_changed"] = (
    pair_suggestions["old_classification"].fillna("").astype(str) !=
    pair_suggestions["new_classification"].fillna("").astype(str)
)

url_pattern = re.compile(r'https?://\S+')

def extract_urls(text):
    if pd.isna(text):
        return set()
    return set(url_pattern.findall(str(text)))

pair_suggestions["old_urls"] = pair_suggestions["old_note_text"].apply(extract_urls)
pair_suggestions["new_urls"] = pair_suggestions["new_note_text"].apply(extract_urls)

def url_change_type(old_urls, new_urls):
    if old_urls == new_urls:
        return "no_change"
    if len(new_urls) > len(old_urls) and old_urls.issubset(new_urls):
        return "add"
    if len(new_urls) < len(old_urls) and new_urls.issubset(old_urls):
        return "remove"
    return "swap"

pair_suggestions["url_change_type"] = pair_suggestions.apply(
    lambda r: url_change_type(r["old_urls"], r["new_urls"]),
    axis=1
)

pair_suggestions[[
    "tweetId",
    "old_noteId",
    "new_noteId",
    "text_changed",
    "classification_changed",
    "url_change_type"
]].head()

,tweetId,old_noteId,new_noteId,text_changed,classification_changed,url_change_type
0,-1,2014117943958012199,2014121093532098991,True,False,swap
1,-1,2014121093532098991,2014466977717485974,True,True,swap
2,-1,2014466977717485974,2014495284156387774,True,False,swap
3,-1,2014495172550406429,2014496979053879586,True,False,swap
4,-1,2014496979053879586,2014716760490901695,True,False,no_change


In [37]:
print("Total (version, suggestion) pairs:", len(pair_suggestions))
print()
print("Text changed:")
print(pair_suggestions["text_changed"].value_counts(dropna=False))
print()
print("Classification changed:")
print(pair_suggestions["classification_changed"].value_counts(dropna=False))
print()
print("URL change type:")
print(pair_suggestions["url_change_type"].value_counts(dropna=False))

Total (version, suggestion) pairs: 4715

Text changed:
text_changed
True     4647
False      68
Name: count, dtype: int64

Classification changed:
classification_changed
False    3704
True     1011
Name: count, dtype: int64

URL change type:
url_change_type
swap         2843
no_change    1219
add           519
remove        134
Name: count, dtype: int64


In [39]:
sample_pairs = version_pairs_df.sample(
    n=100,
    random_state=42
).copy()

sample_pair_suggestions = pair_suggestions.merge(
    sample_pairs[["tweetId", "old_noteId", "new_noteId"]],
    on=["tweetId", "old_noteId", "new_noteId"],
    how="inner"
)

print("Sampled version pairs:", len(sample_pairs))
print("Suggestion rows within sampled version pairs:", len(sample_pair_suggestions))
sample_pair_suggestions.head()

Sampled version pairs: 100
Suggestion rows within sampled version pairs: 66


,tweetId,old_noteId,new_noteId,old_createdAt,new_createdAt,noteId,raterParticipantId,createdAtMillis,suggestion,ratingCreatedAt,old_note_text,old_classification,new_note_text,new_classification,text_changed,classification_changed,old_urls,new_urls,url_change_type
0,1981359374971908312,2023825160823074997,2023917539513045056,2026-02-17 18:24:20.258000+00:00,2026-02-18 00:32:52.488000+00:00,2023825160823074997,DA458F2833E8ECFA21E013D764483F64C8095B53B2492B...,1771374518271,The man in the video is not an asylum seeker; ...,2026-02-18 00:28:38.271000+00:00,No evidence the man is a doctor with master's ...,MISINFORMED_OR_POTENTIALLY_MISLEADING,The man in the video is French comedian Zepeck...,MISINFORMED_OR_POTENTIALLY_MISLEADING,True,False,{https://x.com/globaleeyenews/status/198137432...,{https://www.leparisien.fr/val-d-oise-95/cergy...,swap
1,1981359374971908312,2023825160823074997,2023917539513045056,2026-02-17 18:24:20.258000+00:00,2026-02-18 00:32:52.488000+00:00,2023825160823074997,D3D21AB0B9897B960D8B6F3F20584ABC82E52A65EDA576...,1771376841391,The video shows French social media star Zepec...,2026-02-18 01:07:21.391000+00:00,No evidence the man is a doctor with master's ...,MISINFORMED_OR_POTENTIALLY_MISLEADING,The man in the video is French comedian Zepeck...,MISINFORMED_OR_POTENTIALLY_MISLEADING,True,False,{https://x.com/globaleeyenews/status/198137432...,{https://www.leparisien.fr/val-d-oise-95/cergy...,swap
2,2019600070300319820,2019766181117542405,2019770140523450420,2026-02-06 13:39:03.346000+00:00,2026-02-06 13:56:24.069000+00:00,2019766181117542405,E8CE52D7EFE476D7F637B5F2C0CB4AD83C57D5A79D1636...,1770385701611,Don’t be misandristic,2026-02-06 13:48:21.611000+00:00,Mixed evidence on male loneliness epidemic. Ga...,MISINFORMED_OR_POTENTIALLY_MISLEADING,Mixed evidence on male loneliness epidemic. Ga...,NOT_MISLEADING,True,True,{https://news.gallup.com/poll/690788/younger-m...,{https://news.gallup.com/poll/690788/younger-m...,no_change
3,2020394105382342954,2021325842249638396,2021332849085882657,2026-02-10 20:55:34.629000+00:00,2026-02-10 21:22:40.904000+00:00,2021325842249638396,DA458F2833E8ECFA21E013D764483F64C8095B53B2492B...,1770758280354,When a video or other media is outdated and pr...,2026-02-10 21:18:00.354000+00:00,Moroccan news reports over 400 sub-Saharan mig...,NOT_MISLEADING,Moroccan news reports over 400 sub-Saharan mig...,NOT_MISLEADING,True,False,{https://www.moroccoworldnews.com/2026/02/2779...,{https://www.moroccoworldnews.com/2026/02/2779...,swap
4,2020516480048328786,2020573903509192704,2020584398345949330,2026-02-08 19:03:55.861000+00:00,2026-02-08 19:47:38.747000+00:00,2020573903509192704,DFC29B9D72B82F0015FFC8A917FA406251EA61DF040DAF...,1770579835460,This tweet already explicitly claims private s...,2026-02-08 19:43:55.460000+00:00,The claim that Jason Calacanis visited Epstein...,MISINFORMED_OR_POTENTIALLY_MISLEADING,The post claims private sources confirm Jason ...,MISINFORMED_OR_POTENTIALLY_MISLEADING,True,False,{https://nymag.com/intelligencer/article/who-i...,{https://nymag.com/intelligencer/article/who-i...,swap


In [40]:
llm_input = sample_pair_suggestions[[
    "tweetId",
    "old_noteId",
    "new_noteId",
    "suggestion",
    "old_note_text",
    "new_note_text",
    "old_classification",
    "new_classification",
    "text_changed",
    "classification_changed",
    "url_change_type"
]].copy()

print(llm_input.shape)
llm_input.head()

(66, 11)


,tweetId,old_noteId,new_noteId,suggestion,old_note_text,new_note_text,old_classification,new_classification,text_changed,classification_changed,url_change_type
0,1981359374971908312,2023825160823074997,2023917539513045056,The man in the video is not an asylum seeker; ...,No evidence the man is a doctor with master's ...,The man in the video is French comedian Zepeck...,MISINFORMED_OR_POTENTIALLY_MISLEADING,MISINFORMED_OR_POTENTIALLY_MISLEADING,True,False,swap
1,1981359374971908312,2023825160823074997,2023917539513045056,The video shows French social media star Zepec...,No evidence the man is a doctor with master's ...,The man in the video is French comedian Zepeck...,MISINFORMED_OR_POTENTIALLY_MISLEADING,MISINFORMED_OR_POTENTIALLY_MISLEADING,True,False,swap
2,2019600070300319820,2019766181117542405,2019770140523450420,Don’t be misandristic,Mixed evidence on male loneliness epidemic. Ga...,Mixed evidence on male loneliness epidemic. Ga...,MISINFORMED_OR_POTENTIALLY_MISLEADING,NOT_MISLEADING,True,True,no_change
3,2020394105382342954,2021325842249638396,2021332849085882657,When a video or other media is outdated and pr...,Moroccan news reports over 400 sub-Saharan mig...,Moroccan news reports over 400 sub-Saharan mig...,NOT_MISLEADING,NOT_MISLEADING,True,False,swap
4,2020516480048328786,2020573903509192704,2020584398345949330,This tweet already explicitly claims private s...,The claim that Jason Calacanis visited Epstein...,The post claims private sources confirm Jason ...,MISINFORMED_OR_POTENTIALLY_MISLEADING,MISINFORMED_OR_POTENTIALLY_MISLEADING,True,False,swap


In [43]:
out_path = os.path.join(DATA_DIR, "pair_suggestions_full.parquet")
pair_suggestions.to_parquet(out_path, index=False)

out_path = os.path.join(DATA_DIR, "pair_suggestions_sample100pairs.parquet")
sample_pair_suggestions.to_parquet(out_path, index=False)

out_path = os.path.join(DATA_DIR, "llm_input_sample100pairs.parquet")
llm_input.to_parquet(out_path, index=False)

out_path = os.path.join(DATA_DIR, "llm_input_sample100pairs.csv")
llm_input.to_csv(out_path, index=False)
print("Saved all Step 3 outputs.")

Saved all Step 3 outputs.
